# Snížení redundance senzorů výrobní linky pomocí PROC GVARCLUS

## Shrnutí

Vícezónová výrobní linka proudově generuje desítky vzájemně korelovaných hodnot ze senzorů, z nichž mnohé nesou stejný podkladový signál. Tento notebook používá **PROC GVARCLUS** (shlukování proměnných) k rozdělení 13 procesních senzorů do čtyř disjunktních shluků a poté čte hodnoty R-kvadrát každého shluku, aby vybral jeden reprezentativní snímač na shluk — čímž se sníží rozsah monitorování SPC bez ztráty informace o procesu. Tři ze čtyř shluků se čistě shodují s fyzickými zónami (průměrné R-kvadrát 0,92, 0,93 a 0,96); čtvrtý je jednokanálový shluk, který procedura oddělila samostatně a který v této analýze prozkoumáme, místo abychom ho přešli mlčením.

## Zdroje dat

Všechna data se generují přímo pomocí `call streaminit(20260531)` a `rand()` — bez externích nebo síťových vstupů.

| Dataset | Řádky | Klíčové proměnné | Popis |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Syntetické hodnoty z třízónové výrobní linky. Senzory v rámci jedné zóny sdílejí latentní řídicí signál (vysoká korelace); pro reálnou redundanci jsou přidány mezizónové snímače (teploty vázané na zónu 1, tlaky na zónu 2, vibrace na zónu 3). Smyčka v kroku DATA požaduje 300 cyklů, ale toto testovací prostředí běží bez licence a zachová jen prvních 100 pozorování — to stačí k čistému obnovení shlukové struktury. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | Jeden řádek na vstupní senzor: shluk, do kterého byl přiřazen, a jeho R-kvadrát vůči komponentě daného shluku. Vytvořeno příkazem `OUTPUT OUT=`. |

# Snížení redundance senzorů výrobní linky pomocí PROC GVARCLUS

Na přístrojově vybavené výrobní lince je běžné zaznamenávat desítky senzorů, které měří překrývající se fyzikální jevy — několik termočlánků v jedné zóně, redundantní tlakové převodníky, vibrační sondy sledující stejný hřídel. Přivedení každého kanálu do regulačního diagramu nebo navazujícího modelu plýtvá úsilím při monitorování a zvyšuje multikolinearitu.

**Shlukování proměnných** řeší tento problém přímo. `PROC GVARCLUS` rozděluje číselné proměnné do *disjunktních* shluků, kde je každý shluk shrnut první hlavní komponentou svých členů. Senzory, které se pohybují souběžně, skončí ve stejném shluku; inženýr pak může ponechat jeden reprezentativní kanál na shluk.

V tomto notebooku:

1. Vygenerujeme hodnoty z třízónové linky se záměrně redundantními senzory (smyčka požaduje 300 cyklů; zde je zachováno 100).
2. Shlukujeme 13 kanálů pomocí `PROC GVARCLUS` s `MAXCLUSTERS=4`.
3. Zachytíme přiřazení do shluků ve výstupním datasetu a shrneme je.
4. Interpretujeme hodnoty R-kvadrát a vybereme jeden snímač na shluk pro průběžné SPC.

## Krok 1 — Vygenerování syntetických dat ze senzorů

Simulujeme tři procesní zóny, každou s jedním skrytým řídicím signálem (`zone1_a`, `zone2_a`, `zone3_a`). Zbývající kanály jsou zašuměné lineární funkce řídicího signálu své zóny, takže jsou silně korelované v rámci zóny, ale téměř nezávislé mezi zónami. Zároveň vážeme vstupní/výstupní teploty na zónu 1 a oba tlakové převodníky na zónu 2, což napodobuje mezipřístrojovou redundanci pozorovanou na reálných linkách. Pevný seed zajišťuje reprodukovatelnost běhu. Smyčka požaduje 300 cyklů; v nelicencovaném režimu engine zachová prvních 100 pozorování, což reportuje níže uvedená poznámka NOTE.

In [1]:
data process_sensors;
    CALL streaminit(20260531);
    OPAKUJ cycle = 1 TO 300;
        /* Zona 1: latentni ridici signal plus dve redundantni sondy */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zona 2: latentni ridici signal plus dve redundantni sondy */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zona 3: latentni ridici signal plus jedna redundantni sonda */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Mezipristrojove kanaly vazane na existujici ridici signaly */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        VÝSTUP;
    KONEC;
    ODSTRANIT cycle;
SPUSTIT;



NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.07 seconds
  cpu   0.07 seconds


## Krok 2 — Shlukování senzorů

Voláme `PROC GVARCLUS` na všech 13 kanálech. Příkaz `VAR` vyjmenovává číselné senzory, které se mají shlukovat. Rozdělení omezíme na čtyři shluky pomocí `MAXCLUSTERS=4` (očekáváme zhruba jeden shluk na fyzickou zónu, s malou rezervou). `ODS GRAPHICS ON` s `PLOTS=ALL` zapíná dendrogram shluků proměnných; engine zapisuje toto SVG do pracovního adresáře místo vykreslení přímo v notebooku, takže strukturu, kterou níže čteme, získáváme z vytištěné tabulky **Variable Cluster Assignments** a tabulky vlastních čísel na shluk.

Příkaz `OUTPUT OUT=` zapisuje přiřazení proměnných ke shlukům — spolu s R-kvadrátem každé proměnné vůči vlastnímu shluku — do datasetu, se kterým lze dále programově pracovat.

In [2]:
ODS GRAPHICS ON;

PROCEDURA gvarclus data=process_sensors maxclusters=4 PLOTS=ALL;
    PROMĚNNÁ zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    VÝSTUP out=clusters;
SPUSTIT;

ODS GRAPHICS OFF;



                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Krok 3 — Opětovné spuštění s volbou SUMMARY

Druhé spuštění procedury s volbou `SUMMARY` zachová stejné rozdělení. `PLOTS=NONE` v tomto průchodu vypíná grafiku. V tomto prostředí je vytištěný report totožný s krokem 2 — stejná 13řádková tabulka přiřazení, stejné čtyři shluky a stejný souhrn vlastních čísel — takže tato buňka hlavně ukazuje, že volby `SUMMARY` a `PLOTS=NONE` se správně analyzují a spustí; neočekávají se od ní žádná nová čísla.

In [3]:
PROCEDURA gvarclus data=process_sensors maxclusters=4 summary PLOTS=none;
    PROMĚNNÁ zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
SPUSTIT;



                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Krok 4 — Prozkoumání výstupního datasetu

Dataset `clusters` z `OUTPUT OUT=` má jeden řádek na senzor s jeho přiřazeným `Cluster` a `RSq_Own` (čtvercová korelace mezi senzorem a komponentou jeho shluku). Vysoké `RSq_Own` znamená, že senzor je svým shlukem dobře reprezentován — ideální kandidát na *vyřazení* ve prospěch reprezentanta shluku. Seřadíme podle shluku a poté podle R-kvadrátu, takže nejreprezentativnější senzor v každém shluku bude na vrcholu své skupiny.

In [4]:
PROCEDURA ŘADIT data=clusters out=clusters_ranked;
    PODLE CLUSTER SESTUPNĚ RSq_Own;
SPUSTIT;

PROCEDURA TISK data=clusters_ranked ŠTÍTEK;
    PROMĚNNÁ Variable CLUSTER RSq_Own;
    ŠTÍTEK Variable = "Kanál senzoru"
          CLUSTER  = "Shluk"
          RSq_Own  = "R-kvadrát s vlastním shlukem";
SPUSTIT;



  Obs   Kanál senzoru  Shluk    R-kvadrát s vlastním shlukem
-----  --------------  -----  ------------------------------
    1  ZONE1_A             1                         0.96867
    2  ZONE1_B             1                          0.9189
    3  TEMP_IN             1                        0.903394
    4  TEMP_OUT            1                        0.889519
    5  ZONE2_A             2                         0.98364
    6  ZONE2_B             2                        0.946708
    7  PRESSURE_A          2                        0.929356
    8  PRESSURE_B          2                        0.920915
    9  ZONE2_C             2                        0.892405
   10  ZONE3_A             3                        0.977237
   11  VIBRATION           3                         0.95916
   12  ZONE3_B             3                        0.949054
   13  ZONE1_C             4                               1




NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## Interpretace výsledků

Rozdělení obnovuje většinu fyzické struktury linky, s jednou upřímnou výhradou:

- **Shluk 1** sdružuje `zone1_a` (R²=0,969), `zone1_b` (0,919) a vstupní/výstupní teploty `temp_in` (0,903) a `temp_out` (0,890) — čtyři z pěti kanálů řízených latentním signálem zóny 1. Průměrné R-kvadrát 0,920.
- **Shluk 2** sdružuje všechny tři kanály zóny 2 `zone2_a` (0,984), `zone2_b` (0,947), `zone2_c` (0,892) spolu s oběma tlakovými převodníky `pressure_a` (0,929) a `pressure_b` (0,921), které jsme vázali na řídicí signál zóny 2. Průměrné R-kvadrát 0,935.
- **Shluk 3** sdružuje `zone3_a` (0,977), `zone3_b` (0,949) a `vibration` (0,959). Průměrné R-kvadrát 0,962.
- **Shluk 4** je jediný kanál: `zone1_c`, třetí sonda zóny 1, byla oddělena samostatně s R²=1,000 (jediný člen shluku je triviálně dokonale vysvětlen sám sebou). I přes sdílení řídicího signálu zóny 1 procedura při této velikosti vzorku posoudila `zone1_c` jako dostatečně odlišný od skupiny `zone1_a`/teplot, aby si zasloužil vlastní shluk místo sloučení do shluku 1.

Všechny tři vícekanálové shluky nesou průměrné R-kvadrát nad **0,90**, což potvrzuje, že jedna komponenta vysvětluje většinu variability mezi jejich členy — přesně tu redundanci, kterou chceme sloučit. Osamocený odlehlý případ — `zone1_c` skončivší ve vlastním shluku místo se zbytkem zóny 1 — je užitečnou připomínkou, že shlukování proměnných je řízeno daty: s více cykly nebo vyšším rozpočtem `MAXCLUSTERS` se hranice může posunout, takže rozdělení je výchozím bodem pro inženýrský úsudek, ne pevnou pravdou.

**Doporučení pro linku.** V rámci každého vícekanálového shluku ponechte senzor s nejvyšším `RSq_Own` (kanál nejreprezentativnější pro svůj shluk) a zbytek vyřaďte nebo depriorizujte z běžného sledování SPC — například `zone2_a` pro shluk 2 a `zone3_a` pro shluk 3. Osamocený `zone1_c` berte spíše jako podnět k prošetření než jako automatické zachování: než se rozhodnete jej sledovat samostatně, ověřte, zda nese skutečně odlišnou informaci, nebo jde jen o artefakt shlukování. I s tímto jedním odloženým kanálem se tím sníží 13 sledovaných kanálů na plán se čtyřmi monitorovanými kanály, což omezí šum poplachů a multikolinearitu při zachování informačního obsahu linky.